<a href="https://colab.research.google.com/github/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/blob/rag/rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# import os

# repo_url = "https://github.com/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi.git"
# repo_name = "NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi"

# if os.path.exists("../"+repo_name):
#     print("Repository already present, update...")
#     !git pull
# else:
#     print("Repository clone...")
#     !git clone -b rag {repo_url}
#     %cd {repo_name}

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
# parquet_path = '/content/drive/MyDrive/Progetto-NLP/Branch-rag/collection.parquet'

In [16]:
# import importlib
# import rag
# # from rag import load_parquet
# importlib.reload(rag)

# ds = rag.load_parquet(parquet_patth)

In [17]:
# ds['content']

In [18]:
# import phase
!pip install beir rank_bm25 faiss-cpu hnswlib ir_measures

from rank_bm25 import BM25Okapi
import numpy as np
import faiss
import hnswlib
import time
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import torch
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from datasets import load_dataset


In [19]:
import os
parquet_path = '/content/drive/MyDrive/Progetto-NLP/Branch-rag/collection_ita.parquet'
if (os.path.exists(parquet_path)):
  print("Ok")

ds = load_dataset("parquet", data_files=parquet_path, split="train")

# Get total number of documents without loading them
num_docs = len(ds)
corpus_ids = list(range(num_docs))

print(f"Total documents: {num_docs}")

Ok


Generating train split: 0 examples [00:00, ? examples/s]

Total documents: 8211278


In [20]:
# #This code is useful to update status_registry.json: it moves files names from "mancanti" to "completati" based on their presence or not in the output folder
# import os

# import json

# num_docs=8211278
# totale_count=0
# chunk_size=10000
# output_path="/content/drive/MyDrive/Progetto-NLP/Branch-rag/embedding_collection_ita"

# import re

# def sort_key(filename):
#     # extracts all numbers from filename as integers
#     return [int(x) for x in re.findall(r"\d+", filename)]

# def update_registry(files):
#     """Aggiorna il file JSON tenendo traccia dei file salvati e mancanti."""
#     # 1. Legge il registro esistente
#     registry_file = os.path.join(output_path, "status_registry.json")
#     if os.path.exists(registry_file):
#         with open(registry_file, 'r') as f:
#             registry = json.load(f)
#     else:
#         print("[Error] Register not found!")
#     sorted_files = sorted(files, key=sort_key)
#     print(sorted_files)
#     for filename in sorted_files:
#         #print(filename)
#         # 2. Aggiorna le liste
#         if filename not in registry["completati"]:
#             registry["completati"].append(filename)
#         if filename in registry["mancanti"]:
#             registry["mancanti"].remove(filename)

#     # Calcola la percentuale di completamento totale
#     completati_count = len(registry["completati"])
#     registry["percentuale_completamento"] = f"{(completati_count / totale_count) * 100:.2f}%"

#     # 3. Salva il registro aggiornato su Drive
#     with open(registry_file, 'w') as f:
#         json.dump(registry, f, indent=4)

# def init_registry():
#     registry_file = os.path.join(output_path, "status_registry.json")
#     if os.path.exists(registry_file):
#         with open(registry_file, 'r') as f:
#             registry = json.load(f)
#     else:
#         print("[Error] Register not found!")

#     tutti_i_file_attesi = [
#       f"embeddings_chunk_{i}_{min(i + chunk_size, num_docs)}.pkl"
#       for i in range(0, num_docs, chunk_size)]

#     for filename in tutti_i_file_attesi:
#           #print(filename)
#           # 2. Aggiorna le liste
#           if filename not in registry["mancanti"]:
#               print(filename)
#               registry["mancanti"].append(filename)


# init_registry()

# # for i in range(0, num_docs, chunk_size):
# #       totale_count=totale_count+1

# # from pathlib import Path

# # files=[]
# # folder_path = Path(output_path)
# # for file_path in folder_path.iterdir():
# #     if file_path.is_file():
# #       if file_path.name!="status_registry.json":
# #         #print(file_path.name)
# #         files.append(file_path.name)
# # update_registry(files)

embeddings_chunk_0_10000.pkl
embeddings_chunk_10000_20000.pkl
embeddings_chunk_20000_30000.pkl
embeddings_chunk_30000_40000.pkl
embeddings_chunk_40000_50000.pkl
embeddings_chunk_50000_60000.pkl
embeddings_chunk_60000_70000.pkl
embeddings_chunk_70000_80000.pkl
embeddings_chunk_80000_90000.pkl
embeddings_chunk_90000_100000.pkl
embeddings_chunk_100000_110000.pkl
embeddings_chunk_110000_120000.pkl
embeddings_chunk_120000_130000.pkl
embeddings_chunk_130000_140000.pkl
embeddings_chunk_140000_150000.pkl
embeddings_chunk_150000_160000.pkl
embeddings_chunk_160000_170000.pkl
embeddings_chunk_170000_180000.pkl
embeddings_chunk_180000_190000.pkl
embeddings_chunk_190000_200000.pkl
embeddings_chunk_200000_210000.pkl
embeddings_chunk_210000_220000.pkl
embeddings_chunk_220000_230000.pkl
embeddings_chunk_230000_240000.pkl
embeddings_chunk_240000_250000.pkl
embeddings_chunk_250000_260000.pkl
embeddings_chunk_260000_270000.pkl
embeddings_chunk_270000_280000.pkl
embeddings_chunk_280000_290000.pkl
embeddin

In [21]:

bi_enc = SentenceTransformer('BAAI/bge-m3', model_kwargs={"torch_dtype": torch.float16},)
#bi_enc.max_seq_length = 1024
#x_enc = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def embed_sparse(docs):
    return list(map(lambda text: text.split(" "), docs))

def embed_dense(encoder, docs, batch_size):
    return encoder.encode(docs, convert_to_numpy=True, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)

def dense_search(query, top_k):
    # get similarity scores
    query_embedding = embed_dense(bi_enc, [query])[:1]
    doc_inds, scores = index_hnsw.knn_query(query_embedding, k=top_k)

    # format to run
    run = {corpus_ids[int(ind)]: float(s) for ind, s in zip(doc_inds[0], scores[0])}
    return run

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [22]:
# create dense index processing dataset in chunks and storing them in the drive at the specified path
output_path="/content/drive/MyDrive/Progetto-NLP/Branch-rag/embedding_collection_ita"
bi_enc.max_seq_length = 1024
import os
import pickle
from concurrent.futures import ThreadPoolExecutor

# Assicura che la cartella esista
os.makedirs(output_path, exist_ok=True)

chunk_size = 10000  # Numero di documenti per ogni salvataggio
start_index = 1810000    # Cambia questo valore se devi ripartire da un punto interrotto
stop_index= 2000000

internal_batch_size = 4

def save_to_disk(data, filename):
    with open(os.path.join(output_path, filename), 'wb') as f:
        pickle.dump(data, f)
    print(f"Completato salvataggio: {filename}")

# Sostituiamo len(corpus_docs) con num_docs (che abbiamo definito prima)
with ThreadPoolExecutor(max_workers=2) as executor:
  for i in range(start_index, num_docs, chunk_size):
      end_index = min(i + chunk_size, num_docs)
      if end_index >= stop_index:
        print("Finished embedding the requested slide")
        return

      print(f"Processando blocchi da {i} a {end_index}...")

      # CHANGED HERE: Estraiamo la fetta direttamente dal dataset su disco.
      # Questo carica in RAM SOLO i 10000 documenti di questo blocco.
      batch_docs = ds[i:end_index]['content']

      # Genera gli embedding (usa la tua funzione embed_dense con batch_size=4)
      embeddings = embed_dense(bi_enc, batch_docs, batch_size=internal_batch_size)

      # Salva il pezzetto su Drive
      file_name = f"embeddings_chunk_{i}_{end_index}.pkl"
      executor.submit(save_to_disk, embeddings, file_name)


      # 5. PULIZIA BRUTALE DELLA MEMORIA
      del batch_docs
      del embeddings
      torch.cuda.empty_cache()      # Svuota la VRAM della GPU

print("Tutti i processi di calcolo sono terminati. I salvataggi finali potrebbero richiedere ancora qualche secondo.")

Processando blocchi da 1050000 a 1060000...


Batches:   0%|          | 0/2500 [00:00<?, ?it/s]

Processando blocchi da 1060000 a 1070000...


Batches:   0%|          | 0/2500 [00:00<?, ?it/s]

Completato salvataggio: embeddings_chunk_1050000_1060000.pkl
Processando blocchi da 1070000 a 1080000...
Completato salvataggio: embeddings_chunk_1060000_1070000.pkl


...

Processando blocchi da 1800000 a 1810000...
Completato salvataggio: embeddings_chunk_1790000_1800000.pkl


Batches:   0%|          | 0/2500 [00:00<?, ?it/s]

Processando blocchi da 1810000 a 1820000...
Completato salvataggio: embeddings_chunk_1800000_1810000.pkl


Batches:   0%|          | 0/2500 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import os
import pickle
import numpy as np
import re
import hnswlib
import gc

path = "/content/drive/MyDrive/Progetto-NLP/Branch-rag/embeddings_subset_it"

# 1. Get and sort files
if not os.path.exists(path):
  os.makedirs(path)
files = [f for f in os.listdir(path) if f.endswith('.pkl')]
def extract_number(filename):
    match = re.search(r'chunk_(\d+)_', filename)
    return int(match.group(1)) if match else 0
files.sort(key=extract_number)

# 2. Initialize HNSW Index FIRST
dim = 1024 # BAAI/bge-m3 dimension
index_hnsw = hnswlib.Index(space='cosine', dim=dim)
index_hnsw.init_index(max_elements=num_docs, ef_construction=200, M=32)

# 3. Incrementally load, add to index, and delete from RAM
current_id = 0
for file in files:
    with open(os.path.join(path, file), 'rb') as f:
        chunk_embeddings = pickle.load(f)

    chunk_size = chunk_embeddings.shape[0]
    ids_for_chunk = list(range(current_id, current_id + chunk_size))

    # Add to index
    index_hnsw.add_items(chunk_embeddings, ids_for_chunk)
    print(f"Aggiunto all'indice: {file} (Elementi: {chunk_size})")

    current_id += chunk_size

    # FREE MEMORY explicitly
    del chunk_embeddings
    del ids_for_chunk
    gc.collect()

print("Indice popolato con successo!")

# Initialize the generative component
from transformers import pipeline, AutoTokenizer
pipe = pipeline("text-generation", model="meta-llama/Llama-3.2-1B")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")

In [ ]:
# Initialize the generative component
from transformers import pipeline, AutoTokenizer
from huggingface_hub import login

from google.colab import userdata
TOKEN=userdata.get('HF_TOKEN')

import os

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

pipe = pipeline("text-generation", model="meta-llama/Llama-3.2-1B-Instruct")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")


In [ ]:
#Save complete embeddings at the specified path
index_path = "/content/drive/MyDrive/Progetto-NLP/Branch-rag/embeddings/kurdish_wiki_index.bin"
index_hnsw.save_index(index_path)
print("Indice salvato! Ora puoi respirare.")

In [ ]:

def rag(query, top_k):
    # create system and user messages
    system_prompt = "You are helpful assistant. Answer the question given the supportive information."
    user_prompt = f"Question: {query}"

    # search relevant documents
    if top_k > 0:
        # runs = rrf_search(query, top_k)
        runs = dense_search(query, top_k)
        docs = [ds[int(doc_id)]['content'] for doc_id in runs]
        docs_context = "\n".join(docs)
        user_prompt += f"\n Referenced knowledge: {docs_context}"

    # format prompt
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # generate the answer
    outputs = pipe(
        prompt,
        max_new_tokens=128,
        do_sample=True,
        max_length=None,
        temperature=1.0,
        eos_token_id=pipe.tokenizer.eos_token_id,
        pad_token_id=pipe.tokenizer.pad_token_id
    )
    predicted_answer = outputs[0]['generated_text'][len(prompt):].strip()

    if top_k > 0:
      return prompt, predicted_answer, docs
    else: return prompt, predicted_answer

In [ ]:
  top_k = 0
  prompt, predicted_answer = rag("Chi ha vinto il AEGON Pro Series Edgbaston 2013?", top_k)
  print(f"Prompt: {prompt}\nPredicted answer: {predicted_answer}")

In [ ]:
top_k = 2
prompt, predicted_answer, docs = rag("who was the winner of AEGON Pro Series Edgbaston 2013?", top_k)
i=0
for doc in docs:
  print(f"===============================\nDoc {i}\n:{doc}\n===============================")
  i=i+1
print("==================Answer=============")
print(f"Prompt: {prompt}\nPredicted answer: {predicted_answer}")

In [ ]:
#Debug: test single query vs document
# 1. Prendi il blocco "incriminato" e la tua domanda
target_doc = """<Table>
| 10 cm M>  8 Gebirgshaubitze |
| --- | --- |
M. 10
Tipo: obice da montagna
Origine: Austria-Ungheria
Impiego
Utilizzatori: KuK Armee
Conflitti: prima guerra mondiale
Produzione
Data progettazione: 1908-1910
Date di produzione: 1908-1916
Ritiro dal servizio: 1918
Varianti: 10 cm M. 10 Gebirgshaubitze
Descrizione
Calibro: 104 mm
Peso proiettile: 14,3 kg
Azionamento: otturatore a vite interrotta eccentrico
Elevazione: +43°
voci di armi d'artiglieria presenti su Wikipedia
</Table>
Il 10 cm Gebirgshaubitze M. 8 era un obice da montagna austro-ungarico, impiegato durante la prima guerra mondiale.
"""
query = "Qual è il peso del proiettile 10 cm Gebirgshaubitze M. 8?"

# 2. Genera gli embedding solo per questi due
target_emb = bi_enc.encode(["passage: " + target_doc], normalize_embeddings=True)
query_emb = bi_enc.encode([query], normalize_embeddings=True)

# 3. Calcola la similarità coseno manuale (deve essere alta, es. > 0.7)
similarity = np.dot(query_emb, target_emb.T)
print(f"Similarità tra domanda e blocco specifico: {similarity[0][0]:.4f}")